In [ ]:
%matplotlib ipympl
import sys
import os
sys.path.append("../../../src/")
import matplotlib.pyplot as plt

import cdsaxs
import numpy as np

import cdsaxs.loaders
import cdsaxs.loaders.load_data
import cdsaxs.data
import cdsaxs.plotting
from pathlib import Path

In [ ]:
# set the path to the validation data directory
NOTEBOOK_DIR = Path(os.getcwd())
VALIDATION_DIR = NOTEBOOK_DIR.parents[1] / 'data' / 'test_validation'

In [ ]:
# set the path to the CSV metadata file for the validation data and create the dataset
csv_path = VALIDATION_DIR / 'data_W204_F2' / 'W204_metadata.csv'

In [ ]:
data_dir = VALIDATION_DIR / 'data_W204_F2'
filename_pattern = "W204_F2measure1_{sdd_cm}m_{energy_ev}keV_num{num}_{sample_phi_deg}deg_bpm{bpm}_id{id}_combined.tif"

In [ ]:
dataset = cdsaxs.loaders.load_data.LoadDataset("validation set", data_dir, metadata_pattern=filename_pattern, metadata_scales = {'energy_ev': 1000}, metadata={'exposure_time_s': 2}, filetype='tif')

In [ ]:
# use identical beam center position and sample-to-detector position as the GUI
# set the pixel size for Pilatus detector
dataset.update_all_metadata({'sdd_cm': 504.982, 'center_px': [738, 492], 'pixel_size_um': 172}, overwrite=True)
# dataset.update_all_metadata({'sdd_cm': 504.982, 'center_px': [738, 488], 'pixel_size_um': 172}, overwrite=True) # we are not flipping the images so need to update beam center

In [ ]:
# # rotate all images by 180 degrees to match GUI axis alignments
# for data in dataset.datas.values():
#     data.rotate_image_ccw(2)

# data = dataset.datas["W204_F2measure1_5.2m_16.1keV_num60_00deg_bpm0.417_id857181_combined.tif"]
# fig = data.plot_data()

In [ ]:
data = dataset.datas["W204_F2measure1_5.2m_16.1keV_num60_00deg_bpm0.417_id857181_combined.tif"]
data.metadata

In [ ]:
fig = data.plot_data()

In [ ]:
# normalize by count time
dataset.normalize_all_data_by_metadata(['exposure_time_s'])

In [ ]:
qslice = data.integrate_box('sum', width_qdy_px=5, width_qdx_px=884)

In [ ]:
integrated_dataset = dataset.integrate_dataset('sum', width_qdy_px=5, width_qdx_px=884)
fig = integrated_dataset.plot_data(interpolated_data=True)

In [ ]:
# the slice points (q_values) matches the slices taken in the GUI
q_values = -1*np.array([0.006441, 0.01275, 0.01904, 0.02532, 0.03156, 0.03788, 0.04412, 0.05044, 0.05672, 0.06294, 0.06923, 0.07553, 0.08187, 0.08805, 0.09442, 0.1007, -0.006155, -0.01249, -0.01871, -0.02496, -0.03124, -0.03751, -0.04373, -0.05005, -0.0563, -0.06248, -0.06883, -0.07513, -0.08133, -0.08767, -0.09397, -0.1003]) 
reduced_slices, fig = cdsaxs.reduction.slice_reduced_dataset(integrated_dataset, q_values=q_values, q_widths=0.001, plotting_kwargs={'s': 0.5, 'interpolated_data': True})

In [ ]:
# the dictionary of reduced slices from the refactored code will be referenced as 'refactored_slices'
refactored_slices = {}
for data in reduced_slices.data:
    refactored_slices[float(data.qsx)] = data

In [ ]:
# load the reduced slices from the legacy gui as 'legacy_slices'
legacy_file = VALIDATION_DIR / "LegacyGUISlices_LargerWidthSlices0p001_20250509.csv"
file = open(legacy_file)
data = file.readlines()
file.close()
data = [x.split(',') for x in data]
data = np.array(data, dtype=str)
header = data[0, :]

legacy_slices = {}

for i in range(0, int(header.shape[0]/2)):
    qz = data[1:, i*2]
    iqz = data[1:, i*2+1]

    selection = np.where(qz!='')

    qz = qz[selection].astype(np.float64)
    iqz = iqz[selection].astype(np.float64)

    head = header[i*2+1]
    head = head.split('=')
    head = head[-1]
    if head[-2:] == '\n':
        head = head[:-2]

    qx = np.float64(head)

    legacy_slices[float(np.round(qx, 4))] = cdsaxs.data.data1d.Data1D(q=qz, Iq=iqz, q_axis='qsz')

In [ ]:
%matplotlib inline
plt.close('all')
# plot each slice from the refactored code with the matching slice from the legacy gui
for qx in legacy_slices.keys():
    plt.figure()

    # the coordinate systems differ between old gui and new code so there is a sign flip along qsx and qsz!
    slice_legacy = legacy_slices[qx]
    slice_refactor = refactored_slices[-1*qx]
    
    plt.scatter(-1*slice_refactor.q, slice_refactor.Iq, label='refactored code')
    plt.scatter(slice_legacy.q, slice_legacy.Iq*1, label='legacy gui')

    plt.legend()

    # plt.xlim(-0.001, 0.001)
    
    plt.title(r"q$_x$ = "+str(qx)+r" $\AA^{-1}$")
    plt.ylabel("Intensity")
    plt.xlabel(r"q ($\AA^{-1}$)")
    plt.yscale('log')
    plt.show()
    plt.close()

    # break

In [ ]:
# plot error between the refactored code and legacy gui slices
# the intensity from the refactored code is divided by 2 (the count time in seconds) as this normalization feature is not yet available in the refactored code
%matplotlib inline
all_max_errors = []
for qx in legacy_slices.keys():
    plt.figure()
    slice_legacy = legacy_slices[qx]
    slice_refactor = refactored_slices[-1*qx]

    max_error = 0
    for i, q in enumerate(slice_legacy.q):
        close_match = np.argmin(np.abs(-1*slice_refactor.q - q))
        q_close = -1*slice_refactor.q[close_match]
        # only match up pairs that are sufficiently close in q
        q_error = np.abs((q-q_close)*100/q)
        if q_error <= 0.1:
            error = (slice_refactor.Iq[close_match] - slice_legacy.Iq[i])*100/slice_legacy.Iq[i]
            plt.scatter(q, error)
            if np.abs(error) > np.abs(max_error):
                max_error = error

    all_max_errors.append(max_error)
    # plt.xlim(-0.001, 0.001)
    
    plt.title(r"q$_x$ = "+str(qx)+r" $\AA^{-1}$"+"\nMax Error: "+str(np.abs(max_error))+" %")
    plt.ylabel("Intensity Error (%)")
    plt.xlabel(r"q ($\AA^{-1}$)")
    plt.show()
    plt.close()

    # break

In [27]:
np.max(all_max_errors)

np.float64(9.67742030131122e-06)